# 06 Load Excel Files

* Author: Jeremiah Hansen
* Last Updated: 10/31/2026

This notebook will load data into the `LOCATION` and `ORDER_DETAIL` tables from Excel files.

Works both in Workspaces notebooks and as a deployed warehouse notebook (run by DEMO_DAG): it doesn't rely on `cells.<name>` or SQL result variables, and copies files from the external stage to a temporary internal stage before downloading them.

In [ ]:
# Import python packages
import logging
import pandas as pd

# openpyxl comes from environment.yml in deployed notebooks; install it in Workspaces if missing
try:
    import openpyxl
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openpyxl"])

logger = logging.getLogger("demo_logger")

from snowflake.snowpark.context import get_active_session
session = get_active_session()

# Deployed notebooks run inside DEMO_DB.<ENV>_SCHEMA; Workspaces notebooks may have no database set
if session.get_current_database() is None:
    session.use_database("DEMO_DB")

database_name = session.get_current_database().strip('"')
schema_name = session.get_current_schema().strip('"')

logger.info("06_load_excel_files start")

## Create a function to load Excel worksheet to table

Create a reusable function to load an Excel worksheet to a table in Snowflake.

GET doesn't work on external stages, so each file is first copied into a temporary internal stage with `COPY FILES`, then downloaded locally and processed.

In [ ]:
import os
from openpyxl import load_workbook

session.sql("CREATE TEMPORARY STAGE IF NOT EXISTS EXCEL_TMP_STAGE").collect()

def load_excel_worksheet_to_table_local(session, stage_file_path, worksheet_name, target_table):
  local_directory = "./"
  file_name = os.path.basename(stage_file_path)
  source_dir = os.path.dirname(stage_file_path)

  # Copy file from the external stage to the internal temp stage, then to local storage
  session.sql(f"COPY FILES INTO @EXCEL_TMP_STAGE FROM {source_dir}/ FILES = ('{file_name}')").collect()
  get_status = session.file.get(f"@EXCEL_TMP_STAGE/{file_name}", local_directory)

  with open(f"{local_directory}{file_name}", 'rb') as f:
    workbook = load_workbook(f)
    sheet = workbook[worksheet_name]
    data = sheet.values

    # Get the first line in file as a header line
    columns = next(data)[0:]
    # Create a DataFrame based on the second and subsequent lines of data
    df = pd.DataFrame(data, columns=columns)
 
    df2 = session.create_dataframe(df)
    df2.write.mode("overwrite").save_as_table(target_table)
 
  return True

## Process all Excel worksheets

Loop through each Excel worksheet to process and call our `load_excel_worksheet_to_table_local()` function.

In [ ]:
# Temporary solution to load in the metadata, this should be replaced with a direct query to a directory table (or a metadata table)
files_to_load = session.sql("""
    SELECT '@INTEGRATIONS.FROSTBYTE_RAW_STAGE/intro/order_detail.xlsx' AS STAGE_FILE_PATH, 'order_detail' AS WORKSHEET_NAME, 'ORDER_DETAIL' AS TARGET_TABLE
    UNION
    SELECT '@INTEGRATIONS.FROSTBYTE_RAW_STAGE/intro/location.xlsx', 'location', 'LOCATION'
""").to_pandas()

for index, excel_file in files_to_load.iterrows():
    logger.info(f"Processing Excel file {excel_file['STAGE_FILE_PATH']}")
    load_excel_worksheet_to_table_local(session, excel_file['STAGE_FILE_PATH'], excel_file['WORKSHEET_NAME'], excel_file['TARGET_TABLE'])

logger.info("06_load_excel_files end")

### Debugging

In [ ]:
%%sql -r debug_result
--DESCRIBE TABLE LOCATION;
--SELECT * FROM LOCATION;
SELECT CURRENT_DATABASE(), CURRENT_SCHEMA();